<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "204"
iso_country = "Benin"
year = 2023
import_portfolio = "h6_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_DX.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 19.680*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Benin': 

Number of trade records present in dataset: 13213
Number of partner countries present in dataset: 124
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$3331.3m
Total weight within the dataset: 4156.1 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,204,Benin,792,TÃ¼rkiye,Import,Sea,690590,690590 | Ceramic chimney-pots,6905,"6905 | Roofing tiles, chimn",69,69 | Ceramic products,3735,"Non-refractory ceramic building bricks, floori...",Mining and Quarrying,34,NMM - Mineral products n.e.c.,2728.112,NaN,2728.112,kg,1980.00,1980.0,9818.202277,8883.0,5519.638593,5.419293e+04,['Container'],['Container'],168.608734,167.811090,167.862933,167.862933,168.412171,168.412171,177.439727,343.379979,198.795952,179.318359,180.105064,186.322579,173.252323,322.584399,204.371829,187.282954,183.227839,192.734400,169.583647,356.108469,256.865573,222.790533,206.101888,201.599934,165.957691,378.514522,299.168552,281.586114,218.870367,217.15561,162.398351,386.784439,357.453203,363.353173,238.154055,228.562311,158.772737,379.492068,370.973675,388.109886,285.089912,232.899463,155.306745,338.418329,341.403383,361.129712,275.829279,235.801400,151.848003,299.728454,309.856536,361.545247,292.711840,237.730396,3735
1,1,204,Benin,251,France,Import,Sea,690721,690721 | Ceramic tiles; flags,6907,6907 | Ceramic flags and pa,69,69 | Ceramic products,NaN,NaN,NaN,NaN,NaN,242518.714,NaN,242518.714,mÂ²,18153.68,796077.0,553427.705348,6733.0,4183.690943,2.315370e+06,['all'],['all'],8573.028108,8542.140982,8544.445715,8544.445715,8564.241987,8565.858631,8953.166209,16144.824664,9795.101017,9006.888662,9046.873374,9348.277547,8765.353965,15341.788400,10131.991106,9391.876111,9204.080200,9625.944508,8602.895692,16880.222232,12438.979598,11049.209326,10290.413533,10018.142011,8442.549240,18157.707888,14389.670662,13631.597601,10843.589914,10712.78649,8285.720494,18334.983463,17016.638781,17517.972843,11754.817615,11221.113667,8126.003696,17719.005220,17247.285370,18182.635371,13667.430995,11411.517156,7973.943669,16066.098031,16363.934148,17473.594764,13469.819643,11537.276947,7821.516752,14610.938487,15020.586317,17406.311442,14054.503108,11669.545677,nan


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $1247.7m (37.5%), ... 

                              cifvalue    vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                              
10 | Cereals               679032833.6  1.252871e+09     20.4     30.1
15 | Animal, vegetable or  150717802.0  1.759421e+08      4.5      4.2
02 | Meat and edible meat  124324219.8  6.804490e+07      3.7      1.6
03 | Fish and crustaceans  101639868.4  4.887230e+07      3.1      1.2
19 | Preparations of cere   46882976.7  2.192311e+07      1.4      0.5
17 | Sugars and sugar con   32859109.9  6.779416e+07      1.0      1.6
11 | Products of the mill   31150167.5  6.989386e+07      0.9      1.7
21 | Miscellaneous edible   21449265.3  6.685668e+06      0.6      0.2
20 | Preparations of vege   16842551.6  1.508122e+07      0.5      0.4
22 | Beverages, spirits a    8206441.0  2.172205e+07      0.2      0.5
04 | Dairy produce; birds    7415490.6  3.1496

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $1110.6m (33.3%), ... 

                     cifvalue    vol_kg_vtw  pct_val  pct_vol
eora_desc                                                    
Agriculture       969859921.0  1.448499e+09     29.1     34.9
Food & Beverages  137305021.2  1.375038e+08      4.1      3.3
Fishing             3467766.3  1.558545e+06      0.1      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $1095.5m (32.9%), ... 

                                  cifvalue    vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                                 
PCR - Processed rice           640661929.2  1.112830e+09     19.2     26.8
VOL - Vegetable oils and fats  150577813.4  1.759415e+08      4.5      4.2
OMT - Meat products n.e.c.     128711959.5  6.906883e+07      3.9      1.7
OFD - Food products n.e.c.     121622071.3  1.038778e+08      3.7      2.5
SGR - Sugar                     30475561.8  6.686031e+07      0.9      1.6
MIL - Dairy products             7597998.7  3.047932e+06      0.2      0.1
OAP - Animal products n.e.c.     6923037.9  1.651461e+06      0.2      0.0
V_F - Vegetables, fruit, nuts    5493909.7  1.003873e+07      0.2      0.2
GRO - Cereal grains n.e.c.       1588764.6  6.571925e+06      0.0      0.2
PFB - Plant-based fibers          788406.9  1.509178e+06      0.0      0.0
OCR - Crops n.e.c.               

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 37.5% 
GTAP 32.9% 
EORA 33.3%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46332/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,204,Benin,792,TÃ¼rkiye,Import,Sea,690590,690590 | Ceramic chimney-pots,6905,"6905 | Roofing tiles, chimn",69,69 | Ceramic products,3735,"Non-refractory ceramic building bricks, floori...",Mining and Quarrying,34,NMM - Mineral products n.e.c.,2728.112,NaN,2728.112,kg,1980.00,1980.0,9818.202277,8883.0,5519.638593,5.419293e+04,['Container'],['Container'],168.608734,167.811090,167.862933,167.862933,168.412171,168.412171,177.439727,343.379979,198.795952,179.318359,180.105064,186.322579,173.252323,322.584399,204.371829,187.282954,183.227839,192.734400,169.583647,356.108469,256.865573,222.790533,206.101888,201.599934,165.957691,378.514522,299.168552,281.586114,218.870367,217.15561,162.398351,386.784439,357.453203,363.353173,238.154055,228.562311,158.772737,379.492068,370.973675,388.109886,285.089912,232.899463,155.306745,338.418329,341.403383,361.129712,275.829279,235.801400,151.848003,299.728454,309.856536,361.545247,292.711840,237.730396,3735,NaN
1,1,204,Benin,251,France,Import,Sea,690721,690721 | Ceramic tiles; flags,6907,6907 | Ceramic flags and pa,69,69 | Ceramic products,NaN,NaN,NaN,NaN,NaN,242518.714,NaN,242518.714,mÂ²,18153.68,796077.0,553427.705348,6733.0,4183.690943,2.315370e+06,['all'],['all'],8573.028108,8542.140982,8544.445715,8544.445715,8564.241987,8565.858631,8953.166209,16144.824664,9795.101017,9006.888662,9046.873374,9348.277547,8765.353965,15341.788400,10131.991106,9391.876111,9204.080200,9625.944508,8602.895692,16880.222232,12438.979598,11049.209326,10290.413533,10018.142011,8442.549240,18157.707888,14389.670662,13631.597601,10843.589914,10712.78649,8285.720494,18334.983463,17016.638781,17517.972843,11754.817615,11221.113667,8126.003696,17719.005220,17247.285370,18182.635371,13667.430995,11411.517156,7973.943669,16066.098031,16363.934148,17473.594764,13469.819643,11537.276947,7821.516752,14610.938487,15020.586317,17406.311442,14054.503108,11669.545677,nan,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $944.0m of $3331.3m (28.3%)
Volume: 1491.8 of 4156.1 million kilograms (35.9%)
Value-to-Weight: 0.63 from 0.8 US$ per kg (+-21.1%)
Average distance to market (volume-weighted): 8264.8 from 7113.2 nm (+16.2%)

            food_group  N_com     cifvalue    vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    430  730584324.5  1.199044e+09     21.9     28.9
1        Meat and Prep    104  124358374.9  6.837207e+07      3.7      1.6
2  Unprocessed Cereals     17   38367615.4  1.400358e+08      1.2      3.4
3      Sugar and Honey     69   32866094.1  6.779616e+07      1.0      1.6
4                Dairy     77    7381117.3  3.138289e+06      0.2      0.1
5           Vegetables    186    5904139.8  9.952508e+06      0.2      0.2
6        Fats and Oils     81    3501859.4  2.552008e+06      0.1      0.1
7               Fruits     90    1036270.8  9.063531e+05      0.0      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,5.038041e+06,9.741509e+06,5.642828e+06,5.091093e+06,5.113426e+06,5.289893e+06,4.611598e+06,1.097372e+07,1.014220e+07,1.031102e+07,6.760112e+06,6.487446e+06,4.312505e+06,8.506529e+06,8.793444e+06,1.025957e+07,8.306189e+06,6.747617e+06
1,Fats and Oils,4.920679e+06,9.514537e+06,5.511347e+06,4.972499e+06,4.994316e+06,5.166659e+06,4.504172e+06,1.071817e+07,9.905896e+06,1.007085e+07,6.602625e+06,6.336301e+06,4.212051e+06,8.308460e+06,8.588658e+06,1.002059e+07,8.112586e+06,6.590441e+06
2,Fruits,1.288361e+06,2.484296e+06,1.441684e+06,1.301945e+06,1.307615e+06,1.352510e+06,1.179817e+06,2.799524e+06,2.587435e+06,2.631210e+06,1.727018e+06,1.657449e+06,1.103705e+06,2.173716e+06,2.246212e+06,2.619349e+06,2.119462e+06,1.723919e+06
3,Meat and Prep,1.107192e+08,2.142629e+08,1.240451e+08,1.118914e+08,1.123823e+08,1.162619e+08,1.013336e+08,2.413465e+08,2.230443e+08,2.267258e+08,1.486038e+08,1.426187e+08,9.475040e+07,1.870251e+08,1.933449e+08,2.255977e+08,1.826469e+08,1.483395e+08
4,Processed Cereals,3.529275e+09,6.811794e+09,3.950706e+09,3.565903e+09,3.581484e+09,3.705137e+09,3.231566e+09,7.678366e+09,7.098686e+09,7.217377e+09,4.735762e+09,4.541118e+09,3.022755e+09,5.954284e+09,6.153987e+09,7.178269e+09,5.814872e+09,4.722420e+09
5,Sugar and Honey,1.148092e+08,2.153319e+08,1.273562e+08,1.157454e+08,1.162295e+08,1.202463e+08,1.056321e+08,2.446313e+08,2.270567e+08,2.313945e+08,1.536490e+08,1.459960e+08,9.919885e+07,1.911540e+08,1.970463e+08,2.290481e+08,1.866491e+08,1.515344e+08
6,Unprocessed Cereals,2.513103e+08,4.857557e+08,2.814504e+08,2.539473e+08,2.550594e+08,2.638650e+08,2.300540e+08,5.473363e+08,5.059141e+08,5.143122e+08,3.372666e+08,3.235560e+08,2.151446e+08,4.242753e+08,4.385639e+08,5.116490e+08,4.143443e+08,3.365067e+08
7,Vegetables,1.479367e+07,2.852701e+07,1.655391e+07,1.494714e+07,1.501260e+07,1.552986e+07,1.354766e+07,3.215267e+07,2.972380e+07,3.023002e+07,1.983538e+07,1.903043e+07,1.267333e+07,2.494471e+07,2.578092e+07,3.007221e+07,2.434838e+07,1.979237e+07


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.64,0.08,0.01,0.01,0.03,0.86,0.75,0.77,0.29,0.25,0.57,0.61,0.81,0.54,0.33
1,Fats and Oils,1.31,0.17,0.01,0.02,0.07,1.77,1.54,1.59,0.60,0.52,1.17,1.25,1.66,1.11,0.68
2,Fruits,1.15,0.15,0.01,0.02,0.06,1.56,1.36,1.40,0.53,0.46,1.03,1.10,1.46,0.98,0.60
3,Meat and Prep,0.83,0.11,0.01,0.01,0.04,1.13,0.98,1.01,0.38,0.33,0.74,0.79,1.05,0.71,0.43
4,Processed Cereals,4.49,0.58,0.05,0.07,0.24,6.09,5.29,5.46,2.06,1.79,4.01,4.29,5.69,3.82,2.33
5,Sugar and Honey,3.06,0.38,0.03,0.04,0.17,4.23,3.69,3.83,1.46,1.23,2.80,2.98,3.95,2.66,1.59
6,Unprocessed Cereals,6.11,0.79,0.07,0.10,0.33,8.27,7.19,7.41,2.79,2.44,5.45,5.82,7.73,5.19,3.16
7,Vegetables,2.33,0.30,0.03,0.04,0.12,3.15,2.74,2.83,1.06,0.93,2.08,2.22,2.95,1.98,1.21
